# Desafio 2 - Detecção de anomalias no dataset Pulsar

Detecção de anomalias com LOF (Local Outlier Factor), sem usar a coluna target_class no ajuste. A coluna só é usada depois, para avaliar.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Carregar os dados
df = pd.read_csv('pulsar.csv')

In [ ]:
# Exploração
print(df.shape)
print(df.dtypes)
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Pré-processamento (sem a target_class)
X = df.drop(columns=['target_class'])
X_scaled = StandardScaler().fit_transform(X)

In [ ]:
# LOF
lof = LocalOutlierFactor()
pred = lof.fit_predict(X_scaled)

In [ ]:
# Marcar as anomalias
df['anomalia'] = (pred == -1).astype(int)
df['anomalia'].value_counts()

In [ ]:
# Avaliação com a target_class
print(confusion_matrix(df['target_class'], df['anomalia']))
print(classification_report(df['target_class'], df['anomalia']))

In [ ]:
# Visualização
normais = df[df['anomalia'] == 0]
anomalias = df[df['anomalia'] == 1]
plt.scatter(normais['Mean of the integrated profile'], normais['Excess kurtosis of the integrated profile'], s=5, label='Normal')
plt.scatter(anomalias['Mean of the integrated profile'], anomalias['Excess kurtosis of the integrated profile'], s=5, color='red', label='Anomalia')
plt.xlabel('Mean of the integrated profile')
plt.ylabel('Excess kurtosis of the integrated profile')
plt.legend()
plt.show()

## Resultados

O LOF, com os parâmetros padrão, marcou 151 das 17.898 amostras como anomalias. Comparando com a target_class, só 36 dessas anomalias são pulsares, o que dá um recall de 2% para a classe pulsar.

Ou seja, os pulsares não aparecem como pontos isolados com densidade local diferente dos vizinhos. Eles formam um grupo próprio, e o LOF acaba marcando como anomalia principalmente pontos raros que não são pulsares. Para esse dataset, o LOF sozinho não separa bem as duas classes.